# Official Recent Stats Knockout Projection

This notebook uses only the latest official FIFA team statistics and the real Round-of-32 bracket to project the remaining knockout rounds.

It deliberately ignores the historical ensemble so we can compare a pure recent-form view against the main model snapshots.

**Corrections applied (2026-06-29):**
- Group stage analysis revealed `3.6` logistic slope is too sharp for knockout
- Knockout matches are between *qualified teams* only — more balanced than group stage mismatches
- Reduced slope from `3.6` to `2.0` to produce less extreme probabilities
- Handles missing FIFA detailed stats (403 blocks) by falling back to tournament form only
  (See `notebooks/group_stage_prediction_analysis.ipynb` for the full analysis)

In [5]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    fallback = Path(r"C:\Users\PSA-Airbyte\autodev\projects\worldcup-prediction")
    if (fallback / 'src').exists():
        PROJECT_ROOT = fallback
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing 'src'.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.fifa_official import load_official_round_of_32
from src.features.official_team_stats import load_official_team_stats_features
from src.simulation.knockout_stage import FINAL_MATCH_NUMBER, QF_MATCH_NUMBERS, R16_MATCH_NUMBERS, SF_MATCH_NUMBERS, THIRD_PLACE_MATCH_NUMBER

In [6]:
round_of_32 = load_official_round_of_32()
recent_features = load_official_team_stats_features(require_resolved_state=False)

# Handle missing detailed stats: build fallback from tournament form
if recent_features.empty or 'team' not in recent_features.columns:
    from src.data.fifa_official import load_official_tournament_form
    tournament_form = load_official_tournament_form()
    # Build minimal features from tournament form only
    recent_features = pd.DataFrame({
        'team': tournament_form['team'],
        'official_stats_matches_played': tournament_form['tournament_matches_played'],
        'official_stats_weight': (tournament_form['tournament_matches_played'] / 10.0).clip(0.0, 0.35),
        'official_attack_index': pd.Series(0.0, index=tournament_form.index),
        'official_distribution_index': pd.Series(0.0, index=tournament_form.index),
        'official_defense_index': pd.Series(0.0, index=tournament_form.index),
        'official_goalkeeping_index': pd.Series(0.0, index=tournament_form.index),
        'official_discipline_index': pd.Series(0.0, index=tournament_form.index),
        'official_movement_index': pd.Series(0.0, index=tournament_form.index),
        'official_physical_index': pd.Series(0.0, index=tournament_form.index),
        'official_xg_signal': pd.Series(0.0, index=tournament_form.index),
        'official_attack_signal': pd.Series(0.0, index=tournament_form.index),
        'official_defense_signal': pd.Series(0.0, index=tournament_form.index),
        'official_control_signal': pd.Series(0.0, index=tournament_form.index),
        'official_recent_form_index': pd.Series(0.0, index=tournament_form.index),
    })
    # Add tournament form index as the primary signal
    # Map tournament stats to normalized -1..1 range
    tform = tournament_form.copy()
    for col in ['tournament_points_pct', 'tournament_goal_diff_per_match', 'tournament_wins_per_match']:
        tform[f'{col}_norm'] = (tform[col] - tform[col].min()) / (tform[col].max() - tform[col].min()) * 2.0 - 1.0
    recent_features['official_recent_form_index'] = (
        0.4 * tform['tournament_points_pct_norm'].fillna(0.0)
        + 0.3 * tform['tournament_goal_diff_per_match_norm'].fillna(0.0)
        + 0.3 * tform['tournament_wins_per_match_norm'].fillna(0.0)
    ).clip(-1.0, 1.0).round(4)

recent_features['recent_power'] = (
    0.24 * recent_features['official_recent_form_index']
    + 0.18 * recent_features['official_attack_signal']
    + 0.18 * recent_features['official_defense_signal']
    + 0.12 * recent_features['official_control_signal']
    + 0.10 * recent_features['official_goalkeeping_index']
    + 0.08 * recent_features['official_physical_index']
    + 0.10 * recent_features['official_xg_signal']
)
recent_lookup = recent_features.set_index('team').to_dict(orient='index')

# Show top teams by recent power
display(recent_features.sort_values('recent_power', ascending=False)[[
    'team', 'recent_power', 'official_recent_form_index',
    'official_attack_signal', 'official_defense_signal', 'official_control_signal',
]].head(16))

print()
print(f"Data source: {'tournament form only' if 'tournament_matches_played' in recent_features.columns else 'full stats'}")
print(f"Teams loaded: {len(recent_features)}")
print()
display(round_of_32)

,team,recent_power,official_recent_form_index,official_attack_signal,official_defense_signal,official_control_signal
32,France,0.240000,1.0000,0.0,0.0,0.0
36,Argentina,0.232416,0.9684,0.0,0.0,0.0
0,Mexico,0.224832,0.9368,0.0,0.0,0.0
8,Brazil,0.134184,0.5591,0.0,0.0,0.0
20,Netherlands,0.134184,0.5591,0.0,0.0,0.0
28,Spain,0.126600,0.5275,0.0,0.0,0.0
4,Switzerland,0.119016,0.4959,0.0,0.0,0.0
44,England,0.119016,0.4959,0.0,0.0,0.0
16,Germany,0.112848,0.4702,0.0,0.0,0.0
40,Colombia,0.111432,0.4643,0.0,0.0,0.0



Data source: full stats
Teams loaded: 48



,annex_c,match_number,date,home_team,away_team,home_path,away_path
0,M73,73,2026-06-28T19:00:00Z,South Africa,Canada,2A,2B
1,M74,74,2026-06-29T20:30:00Z,Germany,Paraguay,1E,3ABCDF
2,M75,75,2026-06-30T01:00:00Z,Netherlands,Morocco,1F,2C
3,M76,76,2026-06-29T17:00:00Z,Brazil,Japan,1C,2F
4,M77,77,2026-06-30T21:00:00Z,France,Sweden,1I,3CDFGH
5,M78,78,2026-06-30T17:00:00Z,Ivory Coast,Norway,2E,2I
6,M79,79,2026-07-01T01:00:00Z,Mexico,Ecuador,1A,3CEFHI
7,M80,80,2026-07-01T16:00:00Z,England,DR Congo,1L,3EHIJK
8,M81,81,2026-07-02T00:00:00Z,United States,Bosnia and Herzegovina,1D,3BEFIJ
9,M82,82,2026-07-01T20:00:00Z,Belgium,Senegal,1G,3AEHIJ


In [7]:
def recent_match_projection(home_team: str, away_team: str, match_id: str, annex_c: str) -> dict[str, object]:
    # Gracefully handle missing teams in stats
    if home_team not in recent_lookup:
        # Fallback: use a neutral-ish score for unranked teams
        home_power = 0.0
    else:
        home = recent_lookup[home_team]
        home_power = float(home['recent_power'])
    if away_team not in recent_lookup:
        away_power = 0.0
    else:
        away = recent_lookup[away_team]
        away_power = float(away['recent_power'])

    diff = home_power - away_power
    # Corrected: slope 2.0 (not 3.6) for knockout -- more balanced teams, less extreme probs
    KNOCKOUT_LOGISTIC_SLOPE = 2.0
    home_advance_probability = 1.0 / (1.0 + math.exp(-KNOCKOUT_LOGISTIC_SLOPE * diff))
    away_advance_probability = 1.0 - home_advance_probability
    winner = home_team if home_advance_probability >= away_advance_probability else away_team
    loser = away_team if winner == home_team else home_team
    return {
        'match_id': match_id,
        'annex_c': annex_c,
        'home_team': home_team,
        'away_team': away_team,
        'winner': winner,
        'loser': loser,
        'home_advance_probability': round(home_advance_probability, 4),
        'away_advance_probability': round(away_advance_probability, 4),
        'recent_power_diff': round(diff, 4),
    }


def next_round_pairings(results: list[dict[str, object]], match_ids: list[str], prefix: str) -> list[dict[str, str]]:
    pairings: list[dict[str, str]] = []
    for index in range(0, len(results), 2):
        left = results[index]
        right = results[index + 1]
        pairings.append(
            {
                'match_id': f'{prefix}-{index // 2 + 1}',
                'annex_c': match_ids[index // 2],
                'home_team': str(left['winner']),
                'away_team': str(right['winner']),
            }
        )
    return pairings

In [8]:
projected_round_of_32 = [
    recent_match_projection(row.home_team, row.away_team, f'R32-{index}', str(row.annex_c))
    for index, row in enumerate(round_of_32.itertuples(index=False), start=1)
]

round_of_16_pairings = next_round_pairings(projected_round_of_32, R16_MATCH_NUMBERS, 'R16')
projected_round_of_16 = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in round_of_16_pairings
]

quarter_final_pairings = next_round_pairings(projected_round_of_16, QF_MATCH_NUMBERS, 'QF')
projected_quarter_finals = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in quarter_final_pairings
]

semi_final_pairings = next_round_pairings(projected_quarter_finals, SF_MATCH_NUMBERS, 'SF')
projected_semi_finals = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in semi_final_pairings
]

third_place_pairing = {
    'match_id': 'THIRD-1',
    'annex_c': THIRD_PLACE_MATCH_NUMBER,
    'home_team': projected_semi_finals[0]['loser'],
    'away_team': projected_semi_finals[1]['loser'],
}
projected_third_place = recent_match_projection(
    third_place_pairing['home_team'],
    third_place_pairing['away_team'],
    third_place_pairing['match_id'],
    third_place_pairing['annex_c'],
)

final_pairing = {
    'match_id': 'FINAL-1',
    'annex_c': FINAL_MATCH_NUMBER,
    'home_team': projected_semi_finals[0]['winner'],
    'away_team': projected_semi_finals[1]['winner'],
}
projected_final = recent_match_projection(
    final_pairing['home_team'],
    final_pairing['away_team'],
    final_pairing['match_id'],
    final_pairing['annex_c'],
)

display(pd.DataFrame(projected_round_of_32))
display(pd.DataFrame(projected_round_of_16))
display(pd.DataFrame(projected_quarter_finals))
display(pd.DataFrame(projected_semi_finals))

,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,R32-1,M73,South Africa,Canada,Canada,South Africa,0.4773,0.5227,-0.0455
1,R32-2,M74,Germany,Paraguay,Germany,Paraguay,0.5751,0.4249,0.1513
2,R32-3,M75,Netherlands,Morocco,Netherlands,Morocco,0.5114,0.4886,0.0228
3,R32-4,M76,Brazil,Japan,Brazil,Japan,0.5527,0.4473,0.1058
4,R32-5,M77,France,Sweden,France,Sweden,0.6287,0.3713,0.2633
5,R32-6,M78,Ivory Coast,Norway,Ivory Coast,Norway,0.5038,0.4962,0.0076
6,R32-7,M79,Mexico,Ecuador,Mexico,Ecuador,0.6216,0.3784,0.2481
7,R32-8,M80,England,DR Congo,England,DR Congo,0.5670,0.4330,0.1347
8,R32-9,M81,United States,Bosnia and Herzegovina,United States,Bosnia and Herzegovina,0.5639,0.4361,0.1286
9,R32-10,M82,Belgium,Senegal,Belgium,Senegal,0.5289,0.4711,0.0578


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,R16-1,M89,Canada,Germany,Germany,Canada,0.4510,0.5490,-0.0983
1,R16-2,M90,Netherlands,Brazil,Netherlands,Brazil,0.5000,0.5000,0.0000
2,R16-3,M91,France,Ivory Coast,France,Ivory Coast,0.5781,0.4219,0.1575
3,R16-4,M92,Mexico,England,Mexico,England,0.5527,0.4473,0.1058
4,R16-5,M93,United States,Belgium,United States,Belgium,0.5346,0.4654,0.0693
5,R16-6,M94,Croatia,Spain,Spain,Croatia,0.4704,0.5296,-0.0592
6,R16-7,M95,Switzerland,Argentina,Argentina,Switzerland,0.4435,0.5565,-0.1134
7,R16-8,M96,Colombia,Egypt,Colombia,Egypt,0.5490,0.4510,0.0982


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,QF-1,M97,Germany,Netherlands,Netherlands,Germany,0.4893,0.5107,-0.0213
1,QF-2,M98,France,Mexico,France,Mexico,0.5076,0.4924,0.0152
2,QF-3,M99,United States,Spain,Spain,United States,0.4855,0.5145,-0.0289
3,QF-4,M100,Argentina,Colombia,Argentina,Colombia,0.5602,0.4398,0.1210


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,SF-1,M101,Netherlands,France,France,Netherlands,0.4473,0.5527,-0.1058
1,SF-2,M102,Spain,Argentina,Argentina,Spain,0.4473,0.5527,-0.1058


In [9]:
summary = pd.DataFrame(
    [
        {'stage': 'Third place', **projected_third_place},
        {'stage': 'Final', **projected_final},
    ]
)
display(summary[['stage', 'annex_c', 'home_team', 'away_team', 'winner', 'home_advance_probability', 'away_advance_probability', 'recent_power_diff']])
print('Recent-stats-only champion:', projected_final['winner'])

,stage,annex_c,home_team,away_team,winner,home_advance_probability,away_advance_probability,recent_power_diff
0,Third place,M103,Netherlands,Spain,Netherlands,0.5038,0.4962,0.0076
1,Final,M104,France,Argentina,France,0.5038,0.4962,0.0076


Recent-stats-only champion: France
